# Restraint-guided Chai-1 prediction, worked out example for hcab44

`hcab44` is a &beta;SW-specific antibody 

In [1]:
import sys, pathlib, tempfile, warnings
from Bio import BiopythonWarning

# cif_to_two_chain_pdb() renames the light chain C -> B on purpose, so that
# DockQ scores one antigen-Fv interface. Biopython warns about the collision
# before the old C is gone; the +300 residue offset keeps the numbering unique.
warnings.filterwarnings("ignore", category=BiopythonWarning)
import pandas as pd

sys.path.insert(0, "..")                    # the ekochai package lives one level up
from ekochai import config as C, restraints, scoring, structures, dockq, cdr

ROOT     = pathlib.Path("..").resolve()
REF_DIR  = ROOT / "data" / "reference_structures"
EXP_2CH  = ROOT / "data" / "experimental"          # 2 chains, for DockQ
EXP_3CH  = ROOT / "data" / "experimental_3chain"   # 3 chains, for CDR RMSD
OUTPUTS  = ROOT / "demo_hcab44" / "chai_outputs"
WORK     = pathlib.Path(tempfile.mkdtemp(prefix="hcab44_"))

ANTIBODY = "hcab44"
CUTOFF   = C.PUBLISHED_CUTOFF            # 16 A
SEEDS    = [f"seed_{i}" for i in range(1, 6)]
EPITOPES = C.epitope_scan(ANTIBODY)

print(f"{ANTIBODY}: antigenic region {C.ANTIBODY_REGION[ANTIBODY]},",
      f"{len(EPITOPES)} reference epitopes scanned")
print(" ", " ".join(EPITOPES))

hcab44: antigenic region BSW, 6 reference epitopes scanned
  6mej_C_A_B 6mek_C_F_E 6uyg_E_A_B 6wo5_E_A_B 6woq_E_A_B 8w0y_C_H_L


## 1. Reference epitopes, and restraints 

An antigen residue belongs to a reference epitope if any of its C&alpha; lies
within **8 &Aring;** of an antibody C&alpha; in the aligned reference complex.


In [2]:
rows, restraint_files = [], {}
for ep in EPITOPES:
    pdb_id, ag, hv, lt = ep.split("_")
    out, hits = restraints.build(REF_DIR / f"aligned_{ep}.pdb", ag, hv, lt,
                                 CUTOFF, WORK / f"epitope_{ep}.restraints",
                                 pdb_id=pdb_id)
    restraint_files[ep] = out
    residues = sorted({lab for _, lab in hits}, key=lambda s: int(s[1:]))
    rows.append(dict(epitope=ep, n_restraints=len(hits),
                     n_residues=len(residues), residues=" ".join(residues)))

epitopes = pd.DataFrame(rows)
display(epitopes[["epitope", "n_restraints", "n_residues"]])
for _, r in epitopes.iterrows():
    print(f"{r.epitope:14s} {r.residues}")
print("\n", restraint_files["6mek_C_F_E"].name, "\n")
print(restraint_files["6mek_C_F_E"].read_text())

,epitope,n_restraints,n_residues
0,6mej_C_A_B,13,13
1,6mek_C_F_E,11,11
2,6uyg_E_A_B,21,20
3,6wo5_E_A_B,20,18
4,6woq_E_A_B,22,20
5,8w0y_C_H_L,21,19


6mej_C_A_B     W66 G67 I69 L136 N137 N138 T139 R140 P141 P142 L143 G144 G232
6mek_C_F_E     L136 N137 N138 T139 R140 P141 P142 L143 G144 A163 S192
6uyg_E_A_B     P110 V111 V112 V113 R118 S119 G120 A121 F134 L136 N137 N138 T139 P141 P142 L143 G144 V230 G231 G232
6wo5_E_A_B     P110 V111 V112 V113 R118 S119 G120 A121 F134 L136 N137 N138 T139 R140 P141 P142 L143 G144
6woq_E_A_B     P110 V111 V112 V113 R118 S119 G120 A121 F134 L136 N137 N138 T139 P141 P142 L143 G144 V230 G231 G232
8w0y_C_H_L     S70 P90 C91 R118 S119 G120 A121 F134 L136 N137 N138 T139 R140 P141 P142 L143 S192 G231 G232

 epitope_6mek_C_F_E.restraints 

chainA,res_idxA,chainB,res_idxB,connection_type,confidence,min_distance_angstrom,max_distance_angstrom,comment,restraint_id
B,,A,L136,pocket,1.0,0.0,16.0,antigen-antibody,restraint_0
B,,A,N137,pocket,1.0,0.0,16.0,antigen-antibody,restraint_1
B,,A,N138,pocket,1.0,0.0,16.0,antigen-antibody,restraint_2
B,,A,T139,pocket,1.0,0.0,16.0,antigen-antibody,restraint_3
B,,A,R140,pocket,

## 2. Run Chai-1 

 **Disabled by default (requires installation)**: the models it would
produce are already in `demo_hcab44/chai_outputs/`.

In [3]:
RUN_CHAI = False      # set True to regenerate; needs a GPU and chai_lab

if RUN_CHAI:
    from ekochai import predict
    fasta = ROOT / "data" / "sequences" / f"E2_{ANTIBODY}_input.fasta"
    for seed in SEEDS:
        for ep in EPITOPES:
            predict.run(fasta, WORK / seed / f"epitope_{ep}" / "outputs",
                        seed=int(seed.split("_")[1]),
                        restraints_path=restraint_files[ep])
    SOURCE = WORK
else:
    SOURCE = OUTPUTS
print("models read from", SOURCE.relative_to(ROOT.parent))

models read from Eko_Chai1r/demo_hcab44/chai_outputs


## 3. Score every candidate and select one


Contact-weighted ipTM score. sAB is the mean ipTM between chains A and B,
taken over both directions, and wAB the number of C-alpha contacts within
8 A across that interface.

```
sAB = (A->B + B->A)/2     antigen-heavy        wAB = contacts within 8 A
sAC = (A->C + C->A)/2     antigen-light        wAC = contacts within 8 A
wiPTM = (wAB*sAB + wAC*sAC) / (wAB + wAC)
```


In [4]:
rows = []
for ep in EPITOPES:
    d = SOURCE / "seed_1" / f"epitope_{ep}"
    if not (d / "scores.csv").is_file():
        continue
    for _, r in pd.read_csv(d / "scores.csv").iterrows():
        cif = d / f"pred.model_idx_{int(r.model_idx)}.cif"      # .gz resolved
        if not (cif.exists() or cif.with_suffix(".cif.gz").exists()):
            continue
        w, w_ab, w_ac = scoring.wiptm(r, cif)
        rows.append(dict(epitope=ep, model_idx=int(r.model_idx),
                         iptm=scoring.iptm(r), wiptm=w,
                         contacts_heavy=w_ab, contacts_light=w_ac, cif=cif))
pool = pd.DataFrame(rows)
print(f"{len(pool)} candidates scored ({pool.epitope.nunique()} epitopes"
      f" x {C.N_DIFFUSION_SAMPLES} models)\n")
display(pool.drop(columns="cif").sort_values("wiptm", ascending=False)
            .head(10).round(4))

best = scoring.select_best(pool, metric="wiptm")
print(f"selected: {best.epitope}  model {best.model_idx}  "
      f"wiPTM {best.wiptm:.4f}  (unweighted ipTM {best.iptm:.4f})")
print(f"contacts: heavy {best.contacts_heavy}, light {best.contacts_light}")

# agreement with the published summary
pub = pd.read_csv(ROOT / "data" / "summary" / "chai1r_summary_seeds1to8.csv")
pub = pub[(pub.antibody == ANTIBODY)
          & (pub.condition_label == f"restraints_{CUTOFF}")].set_index("seed")
g = pub.loc["seed_1"]
assert best.epitope == g.reference_epitope_best_wiPTM
assert best.model_idx == g.model_idx_best_wiPTM
assert abs(best.wiptm - g.wipTM_Ab_ag) < 1e-9
print("reproduces the published selection exactly")

30 candidates scored (6 epitopes x 5 models)



,epitope,model_idx,iptm,wiptm,contacts_heavy,contacts_light
5,6mek_C_F_E,0,0.6983,0.7168,52,6
9,6mek_C_F_E,4,0.6963,0.7156,45,4
7,6mek_C_F_E,2,0.6980,0.7143,46,7
8,6mek_C_F_E,3,0.6964,0.7137,49,8
6,6mek_C_F_E,1,0.6967,0.7133,49,8
29,8w0y_C_H_L,4,0.6816,0.7016,45,4
28,8w0y_C_H_L,3,0.6825,0.7010,45,7
27,8w0y_C_H_L,2,0.6826,0.6996,44,7
26,8w0y_C_H_L,1,0.6810,0.6993,50,7
25,8w0y_C_H_L,0,0.6809,0.6991,46,7


selected: 6mek_C_F_E  model 0  wiPTM 0.7168  (unweighted ipTM 0.6983)
contacts: heavy 52, light 6
reproduces the published selection exactly


## 4. Compare with the experimental structure

**DockQ** score. Heavy and light are merged into
one chain first, so DockQ scores a single antigen&ndash;Fv interface.

**CDR-loop C&alpha; RMSD** asks how the paratope sits once the antigen is
aligned. 

In [5]:
records = []
for seed in SEEDS:
    g = pub.loc[seed]
    ep, idx = g.reference_epitope_best_wiPTM, int(g.model_idx_best_wiPTM)
    cif = SOURCE / seed / f"epitope_{ep}" / f"pred.model_idx_{idx}.cif"

    pdb = structures.cif_to_two_chain_pdb(cif, WORK / f"{seed}_selected.pdb")
    score = dockq.run_dockq(pdb, EXP_2CH / f"{ANTIBODY}.pdb") \
            if dockq.dockq_available() else None
    loops, sup = cdr.cdr_rmsd(cif, EXP_3CH / f"{ANTIBODY}_C_H_L.pdb",
                              exp_chains=("C", "H", "L"))

    rec = dict(seed=seed, epitope=ep, model_idx=idx,
               wiPTM=round(g.wipTM_Ab_ag, 4),
               DockQ=score, DockQ_published=g.DockQ_best_wiPTM,
               E2_core_RMSD=round(sup["core_rmsd"], 2))
    rec.update({l["cdr"]: round(l["rmsd"], 2) for l in loops})
    records.append(rec)

comparison = pd.DataFrame(records)
display(comparison)
print(f"wiPTM  spread {comparison.wiPTM.max() - comparison.wiPTM.min():.4f}"
      f"   DockQ  spread "
      f"{comparison.DockQ_published.max() - comparison.DockQ_published.min():.3f}"
      f"   same reference epitope every seed: {comparison.epitope.nunique() == 1}")

,seed,epitope,model_idx,wiPTM,DockQ,DockQ_published,E2_core_RMSD,H1,H2,H3,L1,L2,L3
0,seed_1,6mek_C_F_E,0,0.7168,0.450,0.450,0.73,0.72,2.69,1.84,3.60,3.41,2.73
1,seed_2,6mek_C_F_E,2,0.7143,0.322,0.322,0.76,0.79,2.39,1.48,2.47,2.60,2.08
2,seed_3,6mek_C_F_E,4,0.7121,0.348,0.348,0.80,0.97,2.93,2.06,4.62,4.84,3.41
3,seed_4,6mek_C_F_E,1,0.7141,0.353,0.353,0.76,0.92,2.88,2.25,3.82,4.73,3.08
4,seed_5,6mek_C_F_E,3,0.7116,0.403,0.403,0.68,1.07,2.79,2.30,6.32,6.09,4.10


wiPTM  spread 0.0052   DockQ  spread 0.128   same reference epitope every seed: True
